<a href="https://colab.research.google.com/github/Dodf12/Social-Media-Topic-Modeling/blob/main/RedditTopicModeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installations, Imports, Files



In [ ]:
%pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.7 MB/s eta 0:00:00


In [ ]:
from bertopic import BERTopic
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
negPath = "/content/drive/MyDrive/Twitter/Social Media Analysis of ChatGPT for Therapy/Collected Data/Reddit /reddit_relevant_negative.csv"
posNeuPath = "/content/drive/MyDrive/Twitter/Social Media Analysis of ChatGPT for Therapy/Collected Data/Reddit /reddit_relevant_neu_pos.csv"

In [ ]:
negDF = pd.read_csv(negPath)
posNeuDF = pd.read_csv(posNeuPath)
print(len(negDF))

139


In [ ]:
import ast

posNeuDF['label'] = posNeuDF['sentiment'].apply(lambda x: ast.literal_eval(x)['label'])

reddit_positive_df = posNeuDF[posNeuDF['label'] == 'positive'].reset_index(drop=True)
reddit_neutral_df = posNeuDF[posNeuDF['label'] == 'neutral'].reset_index(drop=True)

# test
print(f"Positive: {len(reddit_positive_df)}")
print(f"Neutral: {len(reddit_neutral_df)}")
#reddit_positive_df.head()
reddit_neutral_df.head()

Positive: 127
Neutral: 521


,Unnamed: 0,Text,sentiment,label
0,0,i private therapy i access real therapy i long...,"{'label': 'neutral', 'score': 0.7493461966514587}",neutral
1,1,respond comment prompt output post others prev...,"{'label': 'neutral', 'score': 0.546159029006958}",neutral
2,2,i good i reddit channel i previous instruction...,"{'label': 'neutral', 'score': 0.6309335231781006}",neutral
3,3,______________________________________________...,"{'label': 'neutral', 'score': 0.5173078775405884}",neutral
4,4,______________________________________________...,"{'label': 'neutral', 'score': 0.5336500406265259}",neutral


In [ ]:
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.cluster import KMeans
from bertopic import BERTopic
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
%pip install gensim
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# BerTopic Negative

In [ ]:
def clean_reddit(file, keywords):
  docs = []
  for string in file["Text"]:
      text = str(string).lower()

      for k in keywords:
        #text = text.replace(k, "")
        text = re.sub(r'\b' + k + r'\b', '', text)

      text = re.sub(r"http\S+|www\.\S+|\b\w+\.(?:ly|com|org|net|io)\S*", " ", text)
      text = re.sub(r"[^\x20-\x7E]", " ", text)
      text = re.sub(r"\b\d+\b", " ", text)
      text = re.sub(r"[|^{}\[\]<>~`@=#]", " ", text)
      text = re.sub(r"\s+", " ", text).strip()
      text = re.sub(r'u201[c-f]|u2018|u2019|u201d|u2026|u00a0', ' ', text)

      docs.append(text)

  # Remove exact duplicates
  docs = list(set(docs))

  # Remove near-duplicates
  seen = set()
  unique_docs = []
  for d in docs:
      words = d.split()
      tail = ' '.join(words[len(words)//5:])
      if tail not in seen:
          seen.add(tail)
          unique_docs.append(d)
  docs = unique_docs

  # Remove short comments
  docs = [d for d in docs if len(d.split()) >= 8]

  docs = [d for d in docs if not re.search(
    r'(moderators subredditif|bot action|discord server free)',
    d, re.IGNORECASE
    )]

  return docs

In [ ]:
custom_stops = list(CountVectorizer(stop_words="english").get_stop_words())
custom_stops += ['like', 'just', 'don', 'use', 'know', 'really',
                 'want', 'think', 've', 'good', 'going', 'new']
custom_stops += ['bot', 'chatbot', 'ai', 'therapist', 'therapists', 'therapy',
                 'gpt', 'mental', 'health', 'person', 'people', 'human', 'humans']
custom_stops += ['u', 'ni', 'd', 's', 't', 'someone', 'something', 'kind']

def run_reddit_negative(file, keywords, n_topics=8):
    vectorizer_model = CountVectorizer(stop_words=custom_stops, min_df=2)
    cluster_model = KMeans(n_clusters=n_topics, random_state=42)
    topic_model = BERTopic(
        embedding_model="all-MiniLM-L6-v2",
        vectorizer_model=vectorizer_model,
        hdbscan_model=cluster_model
    )
    docs = clean_reddit(file, keywords)

    topics, prob = topic_model.fit_transform(docs)
    #topic_model.reduce_topics(docs, nr_topics=n_topics)
    topics = topic_model.topics_

    return topics, prob, topic_model.get_topic_info(), topic_model, docs

In [ ]:
keywords = ['chatgpt', 'chat gpt', 'gpt', 'ai', 'mental', 'health']

r_topics, r_prob, r_topic_info, r_topic_model, r_docs = run_reddit_negative(negDF, keywords, n_topics=3)
r_topic_info

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Topic,Count,Name,Representation,Representative_Docs
0,0,51,0_advice_help_response_professional,"[advice, help, response, professional, life, r...",[i help disorder last night update full page b...
1,1,39,1_adhd_time_self_things,"[adhd, time, self, things, way, likely, anxiet...",[common notion adhd i i underdiagnosed everyon...
2,2,31,2_lot_things_recipe_example,"[lot, things, recipe, example, problem, open, ...",[misinformation big problem erroneous informat...


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=r_topic_info)

https://docs.google.com/spreadsheets/d/1osCRbbA_RxDwF3GDdJQs1nFWRYhmXfw438xIETstyYc/edit#gid=0


In [ ]:
# Validation
def run_bertCoherence(r_topic_model, r_docs):
  topic_words = [[word for word, _ in r_topic_model.get_topic(t)] for t in r_topic_model.get_topics() if t != -1]
  tokenized_docs = [doc.split() for doc in r_docs]
  dictionary = Dictionary(tokenized_docs)
  coh = CoherenceModel(topics=topic_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
  print(f"Coherence: {coh.get_coherence()}")

def run_bertDiversity(r_topic_model):
  unique_words = set()
  all_words = []
  for t in r_topic_model.get_topics():
      if t != -1:
          words = [w for w, _ in r_topic_model.get_topic(t)]
          all_words.extend(words)
          unique_words.update(words)
  print(f"Topic Diversity: {len(unique_words) / len(all_words)}")

def run_bertRandomSeed(docs, n_topics = 5):
  seeds = [42, 123, 456]
  for seed in seeds:
      cluster_model = KMeans(n_clusters=n_topics, random_state=seed)
      topic_model = BERTopic(
          embedding_model="all-MiniLM-L6-v2",
          vectorizer_model=CountVectorizer(stop_words="english", min_df=2),
          hdbscan_model=cluster_model
      )
      topics, prob = topic_model.fit_transform(docs)
      print(f"\nSeed {seed}:")
      for t in topic_model.get_topics():
          if t != -1:
              words = [w for w, _ in topic_model.get_topic(t)][:5]
              print(f"  Topic {t}: {words}")

In [ ]:
run_bertCoherence(r_topic_model, r_docs)
run_bertDiversity(r_topic_model)
run_bertRandomSeed(r_docs, n_topics=5)

Coherence: 0.510334190963636
Topic Diversity: 0.9


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 42:
  Topic 0: ['therapy', 'therapist', 'advice', 'help', 'response']
  Topic 1: ['kind', 'real', 'person', 'people', 'issues']
  Topic 2: ['therapy', 'people', 'human', 'therapist', 'beliefs']
  Topic 3: ['bot', 'recipe', 'example', 'developers', 'thing']
  Topic 4: ['adhd', 'people', 'likely', 'things', 'personalities']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 123:
  Topic 0: ['advice', 'life', 'therapist', 'response', 'time']
  Topic 1: ['therapy', 'therapist', 'real', 'therapists', 'people']
  Topic 2: ['bot', 'recipe', 'example', 'developers', 'people']
  Topic 3: ['therapy', 'people', 'way', 'things', 'therapist']
  Topic 4: ['people', 'therapy', 'beliefs', 'therapeutic', 'pain']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 456:
  Topic 0: ['therapy', 'therapist', 'people', 'real', 'help']
  Topic 1: ['therapy', 'therapist', 'people', 'beliefs', 'answer']
  Topic 2: ['therapy', 'ni', 'advice', 'therapist', 'life']
  Topic 3: ['bot', 'recipe', 'example', 'developers', 'thing']
  Topic 4: ['adhd', 'people', 'likely', 'personalities', 'medication']


# BerTopic Positive

In [ ]:
from google.colab import sheets
pos_topics, pos_prob, pos_topic_info, pos_topic_model, pos_docs = run_reddit_negative(reddit_positive_df, keywords, n_topics=3)
sheets = sheets.InteractiveSheet(df=pos_topic_info)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


https://docs.google.com/spreadsheets/d/146UUqObNvCeQwpBhOucDtFBwLy3pMTJEJy9Ywxn3dKo/edit#gid=0


In [ ]:
# Validation
run_bertCoherence(pos_topic_model, pos_docs)
run_bertDiversity(pos_topic_model)
run_bertRandomSeed(pos_docs, n_topics=3)

Coherence: 0.25116279191817686
Topic Diversity: 0.9


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 42:
  Topic 0: ['people', 'time', 'therapist', 'good', 'week']
  Topic 1: ['therapy', 'therapist', 'good', 'help', 'advice']
  Topic 2: ['therapy', 'human', 'good', 'humans', 'fields']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 123:
  Topic 0: ['people', 'time', 'week', 'good', 'therapist']
  Topic 1: ['therapist', 'therapy', 'good', 'help', 'human']
  Topic 2: ['therapy', 'questions', 'people', 'human', 'better']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 456:
  Topic 0: ['therapy', 'therapist', 'good', 'human', 'great']
  Topic 1: ['time', 'ni', 'therapist', 'week', 'therapy']
  Topic 2: ['people', 'field', 'good', 'information', 'fields']


# Bertopic Neutral

In [ ]:
from google.colab import sheets
neu_topics, neu_prob, neu_topic_info, neu_topic_model, neu_docs = run_reddit_negative(reddit_neutral_df, keywords, n_topics=5)
sheets = sheets.InteractiveSheet(df=neu_topic_info)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


https://docs.google.com/spreadsheets/d/1rDOjJDQr9oS5HwGuEmiip5NxLLu4rcQqTryF_OeYwgI/edit#gid=0


In [ ]:
run_bertCoherence(neu_topic_model, neu_docs)
run_bertDiversity(neu_topic_model)
run_bertRandomSeed(neu_docs, n_topics=5)

Coherence: 0.3695524458525291
Topic Diversity: 0.92


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 42:
  Topic 0: ['adhd', 'questions', 'therapist', 'symptoms', 'user']
  Topic 1: ['therapist', 'therapy', 'good', 'advice', 'therapists']
  Topic 2: ['therapist', 'therapy', 'responses', 'emotions', 'time']
  Topic 3: ['guidance', 'ideas', 'different', 'virtual', 'link']
  Topic 4: ['therapy', 'light', 'pain', 'article', 'privacy']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 123:
  Topic 0: ['therapist', 'therapy', 'good', 'therapists', 'advice']
  Topic 1: ['adhd', 'questions', 'different', 'role', 'user']
  Topic 2: ['therapist', 'responses', 'therapy', 'important', 'thoughts']
  Topic 3: ['link', 'people', 'human', 'society', 'therapy']
  Topic 4: ['therapy', 'data', 'light', 'article', 'pain']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 456:
  Topic 0: ['therapist', 'therapy', 'good', 'therapists', 'advice']
  Topic 1: ['therapist', 'responses', 'therapy', 'thoughts', 'important']
  Topic 2: ['role', 'different', 'guidance', 'ideas', 'questions']
  Topic 3: ['therapy', 'data', 'light', 'article', 'pain']
  Topic 4: ['link', 'people', 'human', 'society', 'time']


# NMF Negative

In [ ]:
def negative_nouns_adj(text):
    is_noun_adj = lambda pos: pos[:2] == 'NN' or pos[:2] == 'JJ'
    tokenized = word_tokenize(text)
    return ' '.join([word for word, pos in pos_tag(tokenized) if is_noun_adj(pos)])

def neg_clean_data_nmf_reddit(file, keywords):
  docs = []
  for string in file["Text"]:
    text = str(string).lower()

    for k in keywords:
        text = text.replace(k, "")

    text = re.sub(r"http\S+|www\.\S+|\b\w+\.(?:ly|com|org|net|io)\S*", " ", text)
    text = re.sub(r"[^\x20-\x7E]", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"[|^{}\[\]<>~`@=#]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r'u201[c-f]|u2018|u2019|u201d|u2026|u00a0|u2019s|u2019t|certn', ' ', text)
    text = re.sub(r"\s+", " ", text).strip()

    string = negative_nouns_adj(text)
    docs.append(string)

  docs = list(set(docs))
  docs = [d for d in docs if len(d.split()) >= 5]

  docs = [d for d in docs if not re.search(
    r'(moderators subredditif|bot action|discord server|repetitive.*comments|theconversation|linkor|ourpublic|perplexity.*bot|firefly.*adobe|generator.*bot|automatic.*tl.*dr|tl.*dr.*shorter)',
    d, re.IGNORECASE
  )]

  return docs


def run_nmf_reddit(file, keys, numTop):
  docs = neg_clean_data_nmf_reddit(file, keys)

  custom_stops = list(TfidfVectorizer(stop_words="english").get_stop_words())
  custom_stops += ['like', 'just', 'don', 'use', 'know', 'really',
                 'want', 'think', 've', 'good', 'going', 'new',
                 'things', 'better', 'need', 'way', 'using', 'help',
                 'does', 'right', 'lot', 'time', 'say', 'feel',
                 'make', 'actually', 'tell', 'chat',
                 'ai', 'chatgpt', 'gpt', 'therapist', 'therapists',
                 'therapy', 'mental', 'health', 'human', 'humans',
                 'people', 'person', 'bot', 'chatbot']

  vectorizer = TfidfVectorizer(
        stop_words='english',
        max_df=0.8,
        min_df=2
  )
  X = vectorizer.fit_transform(docs)

  nmf = NMF(n_components=numTop, random_state=42)
  W = nmf.fit_transform(X)
  H = nmf.components_

  topic_words = []

  vocab = vectorizer.get_feature_names_out()
  for k in range(numTop):
      top_idx = H[k].argsort()[::-1][:10]
      words = [vocab[i] for i in top_idx]
      topic_words.append(words)
  doc_topics = W.argmax(axis=1)
  return doc_topics, topic_words, docs

In [ ]:
keys = ['chatgpt', 'chat gpt', 'gpt', 'ai', 'therapist', 'therapists',
           'therapy', 'mental', 'health', 'human', 'humans']
doc_topics, topic_words, docs = run_nmf_reddit(negDF, keys, 3)
for i, words in enumerate(topic_words):
    print(f"Topic {i}: {words}")

Topic 0: ['person', 'time', 'kind', 'responses', 'wrong', 'professional', 'real', 'help', 'thing', 'way']
Topic 1: ['people', 'adhd', 'things', 'scary', 'real', 'don', 'medication', 'phones', 'article', 'data']
Topic 2: ['advice', 'bullshit', 'important', 'life', 'best', 'able', 'poor', 'suicide', 'care', 'help']


In [ ]:
# Validation
def run_nmfCoherence(topic_words, docs):
    tokenized_docs = [doc.split() for doc in docs]
    dictionary = Dictionary(tokenized_docs)
    coh = CoherenceModel(topics=topic_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
    print(f"NMF Coherence: {coh.get_coherence()}")

def run_nmfDiversity(topic_words):
    unique_words = set()
    all_words = []
    for words in topic_words:
        all_words.extend(words)
        unique_words.update(words)
    print(f"NMF Topic Diversity: {len(unique_words) / len(all_words)}")

def run_nmfRandomSeed(docs, n_topics=3):
    seeds = [42, 123, 456]
    for seed in seeds:
        vectorizer = TfidfVectorizer(stop_words="english", max_df=0.8, min_df=2)
        X = vectorizer.fit_transform(docs)
        nmf = NMF(n_components=n_topics, random_state=seed)
        W = nmf.fit_transform(X)
        H = nmf.components_
        vocab = vectorizer.get_feature_names_out()
        print(f"\nSeed {seed}:")
        for k in range(n_topics):
            top_idx = H[k].argsort()[::-1][:5]
            words = [vocab[i] for i in top_idx]
            print(f"  Topic {k}: {words}")

In [ ]:
run_nmfCoherence(topic_words, docs)
run_nmfDiversity(topic_words)
run_nmfRandomSeed(docs, n_topics=3)

NMF Coherence: 0.4255608692807611
NMF Topic Diversity: 0.9333333333333333

Seed 42:
  Topic 0: ['person', 'time', 'kind', 'responses', 'wrong']
  Topic 1: ['people', 'adhd', 'things', 'scary', 'real']
  Topic 2: ['advice', 'bullshit', 'important', 'life', 'best']

Seed 123:
  Topic 0: ['person', 'time', 'kind', 'responses', 'wrong']
  Topic 1: ['people', 'adhd', 'things', 'scary', 'real']
  Topic 2: ['advice', 'bullshit', 'important', 'life', 'best']

Seed 456:
  Topic 0: ['person', 'time', 'kind', 'responses', 'wrong']
  Topic 1: ['people', 'adhd', 'things', 'scary', 'real']
  Topic 2: ['advice', 'bullshit', 'important', 'life', 'best']


# NMF Positive

In [ ]:
# Only 3 topics since less posts
doc_topics, pos_topic_words, pos_docs = run_nmf_reddit(reddit_positive_df, keys, 3)
for i, words in enumerate(pos_topic_words):
    print(f"Topic {i}: {words}")

Topic 0: ['people', 'things', 'able', 'lot', 'great', 'conversation', 'tool', 'time', 'session', 'thoughts']
Topic 1: ['good', 'way', 'luck', 'answers', 'thank', 'words', 'fields', 'different', 'use', 'time']
Topic 2: ['help', 'decent', 'real', 'haha', 'tips', 'luck', 'weeks', 'solution', 'self', 'better']


In [ ]:
# Validation
run_nmfCoherence(pos_topic_words, pos_docs)
run_nmfDiversity(pos_topic_words)
run_nmfRandomSeed(pos_docs, n_topics=3)

NMF Coherence: 0.3115057981743381
NMF Topic Diversity: 0.9333333333333333

Seed 42:
  Topic 0: ['people', 'things', 'able', 'lot', 'great']
  Topic 1: ['good', 'way', 'luck', 'answers', 'thank']
  Topic 2: ['help', 'decent', 'real', 'haha', 'tips']

Seed 123:
  Topic 0: ['people', 'things', 'able', 'lot', 'great']
  Topic 1: ['good', 'way', 'luck', 'answers', 'thank']
  Topic 2: ['help', 'decent', 'real', 'haha', 'tips']

Seed 456:
  Topic 0: ['people', 'things', 'able', 'lot', 'great']
  Topic 1: ['good', 'way', 'luck', 'answers', 'thank']
  Topic 2: ['help', 'decent', 'real', 'haha', 'tips']


# NMF Neutral

In [ ]:
doc_topics, neu_topic_words, neu_docs = run_nmf_reddit(reddit_neutral_df, keys, 5)
for i, words in enumerate(neu_topic_words):
    print(f"Topic {i}: {words}")

Topic 0: ['people', 'good', 'lot', 'way', 'life', 'able', 'information', 'free', 'things', 'different']
Topic 1: ['questions', 'advice', 'thoughts', 'cognitive', 'user', 'individuals', 'cbt', 'response', 'responses', 'feelings']
Topic 2: ['time', 'bot', 'things', 'conversations', 'able', 'personal', 'responses', 'task', 'tasks', 'real']
Topic 3: ['client', 'session', 'sessions', 'work', 'person', 'treatment', 'report', 'sympathy', 'clients', 'real']
Topic 4: ['character', 'role', 'prompt', 'tony', 'model', 'techniques', 'stay', 'user', 'basic', 'play']


In [ ]:
run_nmfCoherence(neu_topic_words, neu_docs)
run_nmfDiversity(neu_topic_words)
run_nmfRandomSeed(neu_docs, n_topics=5)

NMF Coherence: 0.41617134104366615
NMF Topic Diversity: 0.9

Seed 42:
  Topic 0: ['people', 'good', 'lot', 'way', 'life']
  Topic 1: ['questions', 'advice', 'thoughts', 'cognitive', 'user']
  Topic 2: ['time', 'bot', 'things', 'conversations', 'able']
  Topic 3: ['client', 'session', 'sessions', 'work', 'person']
  Topic 4: ['character', 'role', 'prompt', 'tony', 'model']

Seed 123:
  Topic 0: ['people', 'good', 'lot', 'way', 'life']
  Topic 1: ['questions', 'advice', 'thoughts', 'cognitive', 'user']
  Topic 2: ['time', 'bot', 'things', 'conversations', 'able']
  Topic 3: ['client', 'session', 'sessions', 'work', 'person']
  Topic 4: ['character', 'role', 'prompt', 'tony', 'model']

Seed 456:
  Topic 0: ['people', 'good', 'lot', 'way', 'life']
  Topic 1: ['questions', 'advice', 'thoughts', 'cognitive', 'user']
  Topic 2: ['time', 'bot', 'things', 'conversations', 'able']
  Topic 3: ['client', 'session', 'sessions', 'work', 'person']
  Topic 4: ['character', 'role', 'prompt', 'tony', 'm